# YOLOv7-tiny Drone Detection — Kaggle Training

Fresh Kaggle notebook for training YOLOv7-tiny (PyTorch) from scratch on the `muki2003/yolo-drone-detection-dataset`.

**Model:** YOLOv7-tiny  
**Task:** Object detection  
**Classes:** 1 (`drone`)  
**Input:** 320×320  
**Training:** From scratch  
**Output:** `best.pt`

System and GPU information

In [ ]:
import torch
import sys

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)


Kaggle Input Directory (Datasets) 

In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    level = root.replace("/kaggle/input", "").count(os.sep)
    if level <= 3:
        print("  " * level + os.path.basename(root) + "/")


Finding Drone Dataset path

In [ ]:
from pathlib import Path

dataset_candidates = list(Path("/kaggle/input").rglob("drone_dataset"))
print("Found:", dataset_candidates)

if not dataset_candidates:
    raise FileNotFoundError("Could not find drone_dataset")

dataset = dataset_candidates[0]
print("Dataset root:", dataset)


Drone Dataset divisions paths

In [ ]:
train_images = dataset / "train" / "images"
train_labels = dataset / "train" / "labels"
valid_images = dataset / "valid" / "images"
valid_labels = dataset / "valid" / "labels"

paths = {
    "train images": train_images,
    "train labels": train_labels,
    "valid images": valid_images,
    "valid labels": valid_labels,
}

for name, path in paths.items():
    print(f"{name}: {path}")
    print("  exists:", path.exists())


Dataset counting

In [ ]:
image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def count_images(path):
    return sum(
        1 for f in path.iterdir()
        if f.is_file() and f.suffix.lower() in image_extensions
    )

def count_files(path):
    return sum(1 for f in path.iterdir() if f.is_file())

print("Train images:", count_images(train_images))
print("Train labels:", count_files(train_labels))
print("Valid images:", count_images(valid_images))
print("Valid labels:", count_files(valid_labels))


Checking Label format

In [ ]:
label_files = list(train_labels.glob("*.txt"))
print("Number of label files:", len(label_files))

if label_files:
    print("\nExample:", label_files[0])
    print(label_files[0].read_text())


Copying dataset from input to working directory

In [ ]:
import shutil

working_dataset = Path("/kaggle/working/drone_dataset")

if not working_dataset.exists():
    print("Copying dataset to writable /kaggle/working...")
    shutil.copytree(dataset, working_dataset)
else:
    print("Dataset copy already exists.")

print("Working dataset:", working_dataset)


Verifying copying of dataset

In [ ]:
print("Train images:", count_images(working_dataset / "train" / "images"))
print("Train labels:", count_files(working_dataset / "train" / "labels"))
print("Valid images:", count_images(working_dataset / "valid" / "images"))
print("Valid labels:", count_files(working_dataset / "valid" / "labels"))


Creating data.yaml file for dataset

In [ ]:
data_yaml = f"""train: {working_dataset / "train"}
val: {working_dataset / "valid"}

nc: 1
names:
  - drone
"""

yaml_path = Path("/kaggle/working/drone_data.yaml")
yaml_path.write_text(data_yaml.strip())

print(yaml_path.read_text())


Cloning yolov7 repo

In [ ]:
%cd /kaggle/working
!git clone https://github.com/WongKinYiu/yolov7.git


Installing yolov7 dependencies

In [ ]:
%cd /kaggle/working/yolov7
!pip install -q matplotlib pandas scipy seaborn pyyaml tqdm opencv-python pillow requests psutil gitpython thop tensorboard


Verifying dependencies and environment

In [ ]:
import torch
import cv2
import numpy as np
import yaml

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("OpenCV:", cv2.__version__)
print("NumPy:", np.__version__)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


Configuring yolov7-tiny.yaml file for our dataset

In [ ]:
cfg_path = Path("/kaggle/working/yolov7/cfg/training/yolov7-tiny.yaml")

text = cfg_path.read_text()

print("Before:")
for line in text.splitlines():
    if "nc:" in line:
        print(line)

text = text.replace("nc: 80", "nc: 1")
cfg_path.write_text(text)

print("\nAfter:")
for line in cfg_path.read_text().splitlines():
    if "nc:" in line:
        print(line)


Disabling weight and bias logging to prevent prompt and asking whether you want experiment tracking (For kaggle)

- wandb: (1) Create a W&B account
- wandb: (2) Use an existing W&B account
- wandb: (3) Don't visualize my results
- wandb: Enter your choice:

In [ ]:
import os

os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"

print("W&B disabled.")


Configuring torch behavior to prevent conflict with yolov7

In [ ]:
import os

# YOLOv7 is an older codebase and expects PyTorch's pre-2.6
# torch.load behavior.
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"

print("TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD =",
      os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"])

verifying training file

In [ ]:
%cd /kaggle/working/yolov7
!python train.py --help


Training on 1 epoch without pre-trained weights and images 320x320

In [ ]:
%cd /kaggle/working/yolov7

!python -u train.py \
    --workers 2 \
    --device 0 \
    --batch-size 16 \
    --data /kaggle/working/drone_data.yaml \
    --img 320 320 \
    --cfg cfg/training/yolov7-tiny.yaml \
    --weights "" \
    --name drone-test \
    --hyp data/hyp.scratch.tiny.yaml \
    --epochs 1


Verifying if training successful and weight files generated

In [ ]:
!ls -lh /kaggle/working/yolov7/runs/train/drone-test/weights/


Displaying training results

In [ ]:
from IPython.display import Image, display

display(Image(filename="/kaggle/working/yolov7/runs/train/drone-test/results.png"))


Actual training with 100 epochs without pre-trained weights and images 320x320

In [ ]:
# Actual training: 100 epochs, 320x320, YOLOv7-tiny, from scratch.
%cd /kaggle/working/yolov7

!python -u train.py \
    --workers 2 \
    --device 0 \
    --batch-size 16 \
    --data /kaggle/working/drone_data.yaml \
    --img 320 320 \
    --cfg cfg/training/yolov7-tiny.yaml \
    --weights "" \
    --name yolov7-tiny-drone-100 \
    --hyp data/hyp.scratch.tiny.yaml \
    --epochs 100


Testing trained model

In [ ]:
%cd /kaggle/working/yolov7

!python -u test.py \
    --data /kaggle/working/drone_data.yaml \
    --img 320 \
    --batch 16 \
    --conf 0.001 \
    --iou 0.65 \
    --device 0 \
    --weights runs/train/yolov7-tiny-drone-100/weights/best.pt \
    --name drone-validation


Detecting trained model (running and saving detected object files for all valid folder images)

In [ ]:
%cd /kaggle/working/yolov7

!python detect.py \
    --weights runs/train/yolov7-tiny-drone-100/weights/best.pt \
    --source /kaggle/working/drone_dataset/valid/images \
    --img-size 320 \
    --conf 0.25 \
    --device 0 \
    --name drone-detection


Verifying detected files

In [ ]:
import glob
from IPython.display import Image, display

results = glob.glob(
    "/kaggle/working/yolov7/runs/detect/drone-detection/*"
)

print("Generated files:", len(results))
for file in results[5::-1]:
    print(file)

if results:
    display(Image(filename=results[0]))


Copying best.pt file to working directory

In [ ]:
import shutil

source = Path(
    "/kaggle/working/yolov7/runs/train/yolov7-tiny-drone-100/weights/best.pt"
)
destination = Path("/kaggle/working/weights/yolov7-tiny-drone-100-best.pt")

# Create destination directory if it doesn't exist
destination.parent.mkdir(parents=True, exist_ok=True)

shutil.copy2(source, destination)

print("Saved:", destination)
print("Size:", destination.stat().st_size / (1024 * 1024), "MB")


Getting yolov7 pre-trained weights for fine-tune based training

In [ ]:
!curl -L -o yolov7-tiny.pt "https://github.com/WongKinYiu/yolov7/releases/download/v0.1/yolov7-tiny.pt"

Training with 50 epochs with pre-trained weights and images 320x320

In [ ]:
!python -u train.py \
    --workers 2 \
    --device 0 \
    --batch-size 16 \
    --data /kaggle/working/drone_data.yaml \
    --img-size 320 320 \
    --cfg cfg/training/yolov7-tiny.yaml \
    --weights /kaggle/working/yolov7-tiny.pt \
    --name drone-pretrained-50 \
    --hyp data/hyp.scratch.tiny.yaml \
    --epochs 50

Testing trained model

In [ ]:
%cd /kaggle/working/yolov7

!python -u test.py \
    --data /kaggle/working/drone_data.yaml \
    --img 320 \
    --batch 16 \
    --conf 0.001 \
    --iou 0.65 \
    --device 0 \
    --weights runs/train/drone-pretrained-50/weights/best.pt \
    --name drone-validation


Copying best.pt file to working directory

In [ ]:
import shutil

source = Path(
    "/kaggle/working/yolov7/runs/train/drone-pretrained-50/weights/best.pt"
)
destination = Path("/kaggle/working/weights/drone-pretrained-50-best.pt")

# Create destination directory if it doesn't exist
destination.parent.mkdir(parents=True, exist_ok=True)

shutil.copy2(source, destination)

print("Saved:", destination)
print("Size:", destination.stat().st_size / (1024 * 1024), "MB")


Copying Reparam scripts from input directory (custom dataset) for amb82-mini deployment to yolov7 directory

In [ ]:
import shutil
from pathlib import Path

source = Path("/kaggle/input/datasets/bsef23m509/reparam-amb82")
destination = Path("/kaggle/working/yolov7")

for file in source.iterdir():
    if file.is_file():
        shutil.copy2(file, destination / file.name)
        print("Copied:", file.name)

Fixing Torch for reparam script conflict

In [ ]:
from pathlib import Path

file = Path("/kaggle/working/yolov7/reparam_yolov7-tiny.py")
text = file.read_text()

old = "ckpt = torch.load(args.weights, map_location=device)"
new = "ckpt = torch.load(args.weights, map_location=device, weights_only=False)"

if old in text:
    text = text.replace(old, new)
    file.write_text(text)
    print("Fixed torch.load()")
else:
    print("Could not find the expected line.")

Running Reparam script on best.py

In [ ]:
# Create destination directory if it doesn't exist
!mkdir -p /kaggle/working/weights

!python reparam_yolov7-tiny.py --weights "runs/train/drone-pretrained-50/weights/best.pt" --custom_yaml "yolov7-tiny-deploy.yaml" --output "../weights/drone-pretrained-50-best-reparam.pt"